# Account Ownership Prediction & Outreach Targeting
***Using the Global Findex Database (World Bank)***

- **Goal:** Predict whether an adult has a formal financial account.
Predicting Financial Inclusion Around the World
- **Value:** Enables NGOs/governments to identify financially excluded populations for targeted outreach.
- **Model:** Logistic regression (interpretable) + XGBoost (performance) with SHAP explanations.
- **Deployment** Web dashboard where a policymaker inputs basic profile data and gets exclusion probability + key drivers.
- **Extra:** Country-specific or cross-country comparison for policy briefs.

## Business understanding 
 **why it matters**






## 1. Introduction

### Problem Context

Despite significant progress in global financial inclusion, over **1.4 billion adults remain unbanked**. These individuals are excluded from formal savings, secure payments, credit, and insurance — all of which are essential for economic empowerment.
Understanding **who** remains excluded and **why** is crucial for designing effective outreach programs that connect people to the formal financial system.

### Why This Study

Governments, NGOs, and financial service providers often operate with **limited outreach budgets**. If we can **predict the likelihood** of an adult having (or not having) a formal financial account, we can focus efforts where they will have the **highest impact**.

By combining the **rich demographic, behavioral, and access-related variables** in the Global Findex microdata, we aim to:

* Quantify account ownership patterns across countries and segments
* Identify the **key drivers** of exclusion
* Build a **deployable prediction tool** for outreach targeting

---

## 2. Business Understanding

### Objective

Build an **interpretable** and **accurate** predictive model that estimates an adult’s probability of owning a formal account, highlights top drivers, and enables targeted interventions.

### Guiding Questions

1. Who is most likely to be **unbanked** given their demographic and behavioral profile?
2. Which **factors** most strongly influence account ownership across countries and population subgroups?
3. How can these insights be translated into **actionable outreach strategies** at scale?

### DATA UNDERSTANDING

### 1 Source

* **Global Findex Database (2024 edition)** — World Bank

DATASET URL - 
**https://www.worldbank.org/en/publication/globalfindex/download-data**
* \~300 indicators per adult, covering:

  * Account ownership
  * Payments
  * Saving & borrowing
  * Financial resilience
  * Mobile phone & internet use
  * Digital financial services
  * Demographics (age, gender, education, employment)

### 2. Scope for This Study

We will use **individual-level microdata**, filtering and engineering features relevant to predicting account ownership.




## The Global Findex Database 2025

The Global Findex Database provides almost 300 indicators on topics such as **mobile phone ownership**, **internet use**, **digital safety**, **account ownership**, **payments**, **saving, credit**, and **financial resilience.**
- Global Findex data are reported for all indicators by country, region, and income group. 
- Data are also included summarized by gender, income (adults living in the wealthiest 60% and poorest 40% of households), labor force participation (adults in and out of the workforce), age (young and older adults), and rural and urban residence. Available indicators are reported for 2024, 2021, 2017, 2014, and 2011.

https://www.worldbank.org/en/publication/globalfindex/download-data?utm_source=chatgpt.com

### About the Global Findex 2025

The Global Findex Database is the world's only demand-side survey on financial inclusion and a leading source of data on how adults around the world access and use financial services.


Since its launch in 2011, the Global Findex has provided critical insights into financial inclusion, digital payments, savings, and borrowing behaviors across various economies. The database highlights key trends such as the rise of digital financial services and the gender gap in account ownership. 


The Global Findex 2025 introduces the Digital Connectivity Tracker, a new component that measures access to and use of mobile technology. Combined with financial inclusion data, it offers a holistic view of how mobile infrastructure is expanding access to financial services and improving economic resilience. 


We gratefully acknowledge the financial support and partnership of the Gates Foundation and the Mastercard Foundation, which make the Global Findex database possible.

### Which SDG does the Global Findex focus on?

#### SDG 8: Decent Work and Economic Growth
“Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all.”

##### Target 8.10:

Strengthen the capacity of domestic financial institutions to encourage and expand access to banking, insurance and financial services for all.

##### Other relevant links:

**SDG 1 (No Poverty)** – financial inclusion improves resilience.

**SDG 5 (Gender Equality)** – gender-disaggregated account ownership.

**SDG 9 (Industry, Innovation & Infrastructure)** – through digital financial infrastructure & mobile tech.



## Understand the Core Themes
The dataset focuses on:

1. Financial Inclusion (account ownership, savings, borrowing)

2. Digital Access and Connectivity (mobile ownership, internet use)

3. Economic Resilience (ability to access emergency funds, make digital payments)

### SDG 9 – Industry, Innovation & Infrastructure
 **Goal**: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation.

##### Relevant Columns
- internet, mobileaccount_t_d, con*: Reflect digital infrastructure access

- fin24work, merchant_pay: Reflect how people use tech in commerce/industry

- con30a to con32h: Detailed digital/tech access & usage

- Example Insight for SDG 9:


"In low-income countries, only X% of adults have access to internet or use mobile accounts for transactions, suggesting a digital divide that affects inclusive innovation."

| SDG        | Focus                         | Related Columns                                   |
| ---------- | ----------------------------- | ------------------------------------------------- |
| **SDG 1**  | No Poverty                    | `fin24fam`, `fin24bor`                            |
| **SDG 5**  | Gender Equality               | Compare `account_t_d` by gender                   |
| **SDG 8**  | Decent Work & Economic Growth | `fin24work`, `fin21`                              |
| **SDG 10** | Reduced Inequalities          | `group`, `group2` (poorest 40% vs wealthiest 60%) |


### Next Steps
1. Choose a country or region (e.g. Kenya, SSA, Low-income)

2. Compare trends over time (2011–2024) — use year

3. Analyze disparities by gender, income group, age

4. Visualize:

- Histograms of account ownership

- Bar charts of mobile access by region

- Line charts of internet usage over time

### Which SDG does the Global Findex focus on?

#### SDG 8: Decent Work and Economic Growth
“Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all.”

##### Target 8.10:

Strengthen the capacity of domestic financial institutions to encourage and expand access to banking, insurance and financial services for all.

##### Other relevant links:

**SDG 1 (No Poverty)** – financial inclusion improves resilience.

**SDG 5 (Gender Equality)** – gender-disaggregated account ownership.

**SDG 9 (Industry, Innovation & Infrastructure)** – through digital financial infrastructure & mobile tech.



## DATA PREPROCESSING

1. ### Data loading and undertanding.
- We will first load the dataset **GlobalFindexDatabase2025.csv** 
- Check for missing values through the entire dataset.
- Use 60% as the threshold for missing values, as this will not help in our analysis. Check missingness and choose a strict threshold
- Remove obvious identifiers (country codes, names, IDs) from training to avoid leakage
- Drop columns that are pure duplicates, or duplicates of target variants
- Drop nulls in our target varable **Target variable = account_t_d**
- Fix mixed dtypes (the **DtypeWarning**)
- Drop numeric variables that are nearly perfectly correlated (keep only one representative)



In [38]:
import pandas as pd
import requests

# Fetch the data.
df = pd.read_csv("GlobalFindexDatabase2025.csv")

df.head()

/tmp/ipykernel_105214/1541212024.py:5: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("GlobalFindexDatabase2025.csv")


,countrynewwb,codewb,year,pop_adult,regionwb24_hi,incomegroupwb24,group,group2,account_t_d,fiaccount_t_d,...,con12m_s,con26lm_s,con12w_s,con2f_s,con13_s,con26m_s,con28lm_s,con5a_s,con17c_s,con32h_s
0,Afghanistan,AFG,2011,14575546.0,South Asia (excluding high income),Low income,all,all,0.090050,0.090050,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,ALB,2011,2281010.0,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,0.282681,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Algeria,DZA,2011,26251587.0,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,0.332861,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Angola,AGO,2011,12779501.0,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,0.392035,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Argentina,ARG,2011,30685516.0,Latin America & Caribbean (excluding high income),Upper middle income,all,all,0.331302,0.331302,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
# Display the shape and columns of the DataFrame
print("shape:", df.shape)
print("\ncolumns:\n", df.columns.tolist())

# summary: dtype / missing / unique
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_missing': df.isnull().sum(),
    'pct_missing': df.isnull().mean(),
    'n_unique': df.nunique(dropna=False)
})
summary = summary.sort_values('pct_missing', ascending=False)
summary.head(50)   # show top 40 by missingness


shape: (8566, 437)

columns:
 ['countrynewwb', 'codewb', 'year', 'pop_adult', 'regionwb24_hi', 'incomegroupwb24', 'group', 'group2', 'account_t_d', 'fiaccount_t_d', 'mobileaccount_t_d', 'borrow_any_t_d', 'fin4_d', 'dig_acc', 'fin11_2a', 'fin11a', 'fin11b', 'fin11c', 'fin11f', 'fin11d', 'fin11e', 'fin14a', 'fin14b', 'fin14c', 'fin14d', 'fin13_1a', 'fin13_1b', 'fin26a', 'fin26b', 'fin27a', 'fin27b', 'fin17f', 'fin17a_17a1_d', 'fin17a', 'fin17b', 'fin17c', 'fin22d', 'fin22e', 'fin22a_22a1_22g_d', 'fin22a', 'fin22a_1', 'fin22b', 'fin22c', 'fin24sav', 'fin24fam', 'fin24work', 'fin24bor', 'fin24sell', 'fin24other', 'fin24aVD', 'fin24aSD', 'fin24aND', 'fin24aSD_ND', 'fin24aP', 'fin24aN', 'fin24sav_SD_ND', 'fin24fam_SD_ND', 'fin24work_SD_ND', 'fin24bor_SD_ND', 'fin24sell_SD_ND', 'fin24other_SD_ND', 'fin24sav_VD', 'fin24fam_VD', 'fin24work_VD', 'fin24bor_VD', 'fin24sell_VD', 'fin24other_VD', 'fh1', 'fin28', 'fh2', 'fin29', 'fin31a_31b', 'fin30', 'fin31a', 'fin31b', 'fin31d', 'fin32_33_34a', 'fi

,dtype,n_missing,pct_missing,n_unique
con32h_s,float64,8565,0.999883,2
con13_s,float64,8565,0.999883,2
fin37_38_39d_s,float64,8565,0.999883,2
fin32_33_34b_s,float64,8565,0.999883,2
con12m_s,float64,8565,0.999883,2
fing2p_card_s,float64,8565,0.999883,2
con26m_s,float64,8565,0.999883,2
con17c_s,float64,8565,0.999883,2
con28lm_s,float64,8565,0.999883,2
con5a_s,float64,8565,0.999883,2


In [40]:
print(df.isnull().sum().sum())


3294952


In [41]:

# assume df already loaded
df1 = df.copy()

# 1. Keep rows with non-missing target
df1 = df[~df['account_t_d'].isna()].reset_index(drop=True)

# 2. Remove columns with >60% missing (tunable threshold)
threshold = 0.60
high_missing = df1.columns[df1.isnull().mean() > threshold].tolist()
len(high_missing), high_missing[:10]

df1 = df1.drop(columns=high_missing)
print("shape after dropping high-missing cols:", df1.shape)


shape after dropping high-missing cols: (8476, 38)


In [23]:
df1.shape

(8476, 38)

In [42]:
# Display the shape and columns of the DataFrame
print("shape:", df1.shape)
print("\ncolumns:\n", df1.columns.tolist())

# summary: dtype / missing / unique
summary = pd.DataFrame({
    'dtype': df1.dtypes.astype(str),
    'n_missing': df1.isnull().sum(),
    'pct_missing': df1.isnull().mean(),
    'n_unique': df1.nunique(dropna=False)
})
summary = summary.sort_values('pct_missing', ascending=False)
summary.head(50)   # show top 40 by missingness


shape: (8476, 38)

columns:
 ['countrynewwb', 'codewb', 'year', 'pop_adult', 'regionwb24_hi', 'incomegroupwb24', 'group', 'group2', 'account_t_d', 'fiaccount_t_d', 'borrow_any_t_d', 'fin17a_17a1_d', 'fin17a', 'fin22d', 'fin22a_22a1_22g_d', 'fin22a', 'fin22b', 'fin24aP', 'fin24aN', 'fin31a_31b', 'fin30', 'fin31d', 'fin32_n33_acc', 'fin32_n33', 'fin32', 'fin32_acc', 'fin34a', 'fin37_38', 'fin37', 'fin2_t_d', 'fin42', 'fin10', 'fing2p_fin', 'fing2p', 'g20_made', 'g20_received', 'g20_any', 'save_any_t_d']


,dtype,n_missing,pct_missing,n_unique
fin42,float64,5059,0.596862,3322
fin37_38,float64,4943,0.583176,3533
fin32_n33_acc,float64,4941,0.582940,3532
fing2p_fin,float64,4859,0.573266,3616
fin37,float64,4613,0.544243,3863
fin22d,float64,4539,0.535512,3937
fin31a_31b,float64,4537,0.535276,3937
fin34a,float64,4531,0.534568,3944
fing2p,float64,4457,0.525838,4019
fin10,float64,4265,0.503185,4163


In [43]:
df1.columns.tolist()[:38]  # show first 10 columns after cleaning

['countrynewwb',
 'codewb',
 'year',
 'pop_adult',
 'regionwb24_hi',
 'incomegroupwb24',
 'group',
 'group2',
 'account_t_d',
 'fiaccount_t_d',
 'borrow_any_t_d',
 'fin17a_17a1_d',
 'fin17a',
 'fin22d',
 'fin22a_22a1_22g_d',
 'fin22a',
 'fin22b',
 'fin24aP',
 'fin24aN',
 'fin31a_31b',
 'fin30',
 'fin31d',
 'fin32_n33_acc',
 'fin32_n33',
 'fin32',
 'fin32_acc',
 'fin34a',
 'fin37_38',
 'fin37',
 'fin2_t_d',
 'fin42',
 'fin10',
 'fing2p_fin',
 'fing2p',
 'g20_made',
 'g20_received',
 'g20_any',
 'save_any_t_d']

In [44]:
df1.head()  # display the first few rows of the cleaned DataFrame

,countrynewwb,codewb,year,pop_adult,regionwb24_hi,incomegroupwb24,group,group2,account_t_d,fiaccount_t_d,...,fin37,fin2_t_d,fin42,fin10,fing2p_fin,fing2p,g20_made,g20_received,g20_any,save_any_t_d
0,Afghanistan,AFG,2011,14575546.0,South Asia (excluding high income),Low income,all,all,0.090050,0.090050,...,NaN,0.047083,NaN,0.008257,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,ALB,2011,2281010.0,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,0.282681,...,NaN,0.211240,NaN,0.105931,NaN,NaN,NaN,NaN,NaN,NaN
2,Algeria,DZA,2011,26251587.0,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,0.332861,...,NaN,0.135368,NaN,0.011603,NaN,NaN,NaN,NaN,NaN,NaN
3,Angola,AGO,2011,12779501.0,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,0.392035,...,NaN,0.297585,NaN,0.154860,NaN,NaN,NaN,NaN,NaN,NaN
4,Argentina,ARG,2011,30685516.0,Latin America & Caribbean (excluding high income),Upper middle income,all,all,0.331302,0.331302,...,NaN,0.298497,NaN,0.219409,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
see = df1['group2'].value_counts().reset_index()
see.columns = ['group2', 'count']
see = see.sort_values('count', ascending=False)
print(see.shape)
print(see.head())  # display the first few rows of the 'group2' column
print(see['group2'].unique())  # display unique values in the 'group2' column
see.tail()  # display the last few rows of the 'group2' column

(13, 2)
       group2  count
0         all    771
1         men    745
2       women    745
3     age 25+    745
4  ages 15-24    745
['all' 'men' 'women' 'age 25+' 'ages 15-24' 'secondary edu or more'
 'prim edu or less' 'out of laborforce' 'in laborforce' 'richest 60%'
 'poorest 40%' 'urban' 'rural']


,group2,count
8,in laborforce,735
9,richest 60%,732
10,poorest 40%,732
11,urban,152
12,rural,151


In [46]:
see1 = df1['group'].value_counts().reset_index()
see1.columns = ['group', 'count']
see1 = see1.sort_values('count', ascending=False)
print(see1.shape)
print(see1.head())  # display the first few rows of the 'group' column
print(see1['group'].unique())  # display unique values in the 'group' column
see.tail()  # display the last few rows of the 'group' column

(7, 2)
        group  count
0      gender   1490
1     age_cat   1490
2   education   1488
3  laborforce   1470
4      income   1464
['gender' 'age_cat' 'education' 'laborforce' 'income' 'all' 'urbanicity']


,group2,count
8,in laborforce,735
9,richest 60%,732
10,poorest 40%,732
11,urban,152
12,rural,151
